In [ ]:
import uuid
import os
import json
from typing import Dict, List, Any
import requests
from pathlib import Path
import time


In [ ]:
#!/usr/bin/env python3
from __future__ import annotations

import json
import os
import random
import time
from dataclasses import dataclass, asdict
from typing import Dict, Any, List, Optional

# ---- Configuration -----------------------------------------------------------

DEFAULT_BASE_URL = os.getenv("LMSTUDIO_BASE_URL", "http://127.0.0.1:1234")
DEFAULT_MODEL = os.getenv("LMSTUDIO_MODEL", "model-name-here")
DEFAULT_API_KEY = os.getenv("LMSTUDIO_API_KEY", "not-needed")   # LM Studio ignores API key


TEMPERATURE = 0.7
MAX_TOKENS = 4094

# Per-task overrides 
TASK_CONFIG = {
    # Call 1: many topics, 50–80 words each + summary
    "call1": {"max_tokens": 6500, "timeout": 300},

    # Call 2: scales – still potentially long, but less narrative
    "call2_maes": {"max_tokens": 2500, "timeout": 300},
    "call2_amas": {"max_tokens": 2500, "timeout": 300},
    "call2_mseaq": {"max_tokens": 3500, "timeout": 420},

    # Call 3: forma mentis, many cue words but shorter snippets
    "call3": {"max_tokens": 5000, "timeout": 300},

    # Call 4: MCQ problem solving
    "call4": {"max_tokens": 6500, "timeout": 300},
}



def new_run_id() -> str:
    """Generate a short run id to link the 4 calls for the same persona."""
    return uuid.uuid4().hex[:10]


def call_llm_with_grammar_task(task: str, messages: List[Dict[str, str]]) -> str:
    """
    Call LM Studio using GBNF grammar for a specific task.
    Uses the minimal JSON grammar defined in GRAMMARS[task].
    """
    if task not in GRAMMARS:
        raise KeyError(f"No grammar defined for task {task!r}")

    grammar_text = GRAMMARS[task]

    payload = {
        "model": DEFAULT_MODEL,
        "messages": messages,
        "temperature": TEMPERATURE,        # deterministic for grammar
        "max_tokens": MAX_TOKENS,
        "grammar": grammar_text,
        "grammar_type": "gbnf",
    }

    url = f"{DEFAULT_BASE_URL}/chat/completions"
    print(f"🧩 [Grammar] POST {url}  (task={task})")

    resp = requests.post(url, json=payload, timeout=180)
    resp.raise_for_status()

    data = resp.json()
    content = data["choices"][0]["message"]["content"]
    if not content or not content.strip():
        raise RuntimeError(f"Empty response for task {task}")
    return content.strip()

def safe_export_json(data: dict, path: str) -> bool:
    """
    Try to export JSON. Returns True if successful, False if failed.
    Prevents incomplete or corrupted JSON files.
    """
    try:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        return True
    except Exception as e:
        print(f"❌ JSON export failed for {path}: {e}")
        return False

# ---- Data & Randomization ----------------------------------------------------

CITIES = [
    "Rome", "Milan", "Naples", "Bari", "Lecce", "Bologna",
    "New York", "Los Angeles", "Kansas City", "Philadelphia", "Chicago", "Washington DC" 
]

CITIES_ITA = [
    "Rome", "Milan", "Naples", "Bari", "Lecce", "Bologna"
]

CITIES_USA = [
    "New York", "Los Angeles", "Kansas City", "Philadelphia", "Chicago", "Washington DC" 
]

MIGRATION_STATUS = [
    "native-born Italian", "native-born American", "immigrant" 
]

RELIGION = [
    "Christianity", "Islam", "Christianity", "Islam", "Christianity", "Islam", "Christianity", "Islam", "Hinduism", "Hinduism", "Buddhism", "Judaism", "Atheist", "Atheist", "Agnostic" 
]

GENDER_IDENTITY = ["man", "woman"]

SEXUAL_IDENTITY = ["heterosexual","heterosexual","heterosexual","heterosexual","heterosexual", "homosexual", "bisexual", "asexual"]

SUBJECT_POOL = [
    "maths", "physics", "chemistry", "biology", "computer science",
    "philosophy", "history", "art", "economics", "literature"
]

EMPLOYMENT_STATUS = [
    "employed full-time", 
    "employed part-time", "self-employed", "student",
    "employed part-time", "self-employed", "student",
    "employed part-time", "self-employed", "student",
    "student", "student", "student", "student",
    "unemployed", "retired",
    "unemployed", "retired"
]

MID_EDU_LEVELS = [
    "no formal education", "primary school", "lower secondary", "upper secondary",
    "vocational diploma", "bachelor's degree", "master's degree"
]

LOW_EDU_LEVELS = [
    "no formal education", "primary school", "lower secondary", "upper secondary",
    "vocational diploma"
]

# Education levels considered
ALL_EDU_LEVELS = [
    "no formal education", "primary school", "lower secondary", "upper secondary",
    "vocational diploma", "bachelor's degree", "master's degree", "PhD"
]

UNI_EDU_LEVELS = [
    "bachelor's degree", "master's degree", "PhD"
]

MARITAL_STATUSES = ["single", "in a relationship", "married", "divorced", "widowed"]

HOBBIES = [
    "hiking", "reading novels", "baking", "football (calcio)", "running", "yoga",
    "board games", "photography", "gardening", "volunteering", "cinema",
    "classical music", "painting", "cycling", "videogames", "travel", "cooking",
    "birdwatching", "knitting", "rock climbing"
]

TOPIC_POOL = [
    "What is your relationship with mathematics?",
    "Do you ever get anxious when thinking about mathematics?",
    "Did you ever use AI to support your math learning in the last year? If yes, how was your experience?",
    "How would you explain, step by step, how to solve a second order algebraic equation?",
    "How would you explain, step by step, how to find the stationary points of an equation y=f(x)?",
    "Briefly, how do you perform a Principal Component Analysis? Should I get anxious about its mathematics? Please, teach me.",
    "According to you, how can LLMs be used to innovate math learning in schools and universities?"
]

ROLE_MODES = ["human", "human"]  # "human" => role-play persona, "llm" => speak as an AI assistant

FORMA_MENTIS_CUES = [
    "mathematics", "equation", "numbers", "theorem", "proof", #math domain knowledge
    "informatics", "algorithm", "computation", "problem-solving", "variable", #computational thinking
    "AI", "LLM", "model", "ChatGPT", "data", #artificial intelligence
    "exam", "grade", "homework", "failure", "success", #academic assessment
    "class", "lecture", "study", "classroom", "blackboard", #academic context
    "job", "career", "work", "society", "future", #work context
    "STEM", "science", "physics", "chemistry", "biology", #STEM fields
    "art", "music", "literature", "history", "philosophy", #non-STEM fields
    "creativity", "experiment", "logic", "anxiety", "teamwork", #skills
    "professor", "teacher", "student", "knowledge", "scientist" #actors
]

def choose_ocean() -> Dict[str, Dict[str, Any]]:
    """
    Generate OCEAN trait scores (0-100) plus categorical descriptor.
    """
    def bucket(score: int) -> str:
        if score < 33: return "low"
        if score > 66: return "high"
        return "moderate"

    traits = {}
    for key in ["openness", "conscientiousness", "extraversion", "agreeableness", "neuroticism"]:
        score = random.randint(0, 100)
        traits[key] = {"score": score, "level": bucket(score)}
    return traits


def choose_children(age: int, marital_status: str, sexual_orientation: str) -> int:
    """
    Number of children depends on age and marital status.
    For asexual individuals, having children remains possible,
    but is somewhat less likely on average.

    """
    if marital_status == "single":  
        if age < 22:
            choices = [0, 1]
            weights = [20, 1]
        elif age < 30:
            choices = [0, 1, 2]
            weights = [10, 2, 0.5]
        elif age < 40:
            choices = [0, 1, 2]    
            weights = [5, 4, 2]
        else:
            choices = [0, 1, 2]  
            weights = [4, 4, 3]

    elif marital_status == "in a relationship":
        if age < 22:
            choices = [0, 1]
            weights = [15, 0.5]
        elif age < 30:
            choices = [0, 1, 2]
            weights = [6, 4, 1]
        elif age < 40:
            choices = [0, 1, 2]   
            weights = [2, 4, 4]
        else:
            choices = [0, 1, 2] 
            weights = [2, 3, 4]

    elif marital_status == "married":
        if age < 22:
            choices = [0, 1]
            weights = [8, 0.5]
        elif age < 30:
            choices = [0, 1, 2]
            weights = [3, 4, 3]
        elif age < 40:
            choices = [0, 1, 2]    
            weights = [1.5, 3, 4]
        else:
            choices = [0, 1, 2]
            weights = [1.5, 3, 4]

    else:  # divorced / widowed
        if age < 30:
            choices = [0, 1, 2]
            weights = [4, 3, 1]
        elif age < 40:
            choices = [0, 1, 2] 
            weights = [2, 4, 4]
        else:
            choices = [0, 1, 2]
            weights = [1.5, 3, 4]

    if sexual_orientation == "asexual":    
        adjusted_weights = []
        for c, w in zip(choices, weights): 
            if c == 0:
                adjusted_weights.append(w * 1.5)
            elif c == 1:
                adjusted_weights.append(w * 0.8)
            else:
                adjusted_weights.append(w * 0.6)
        weights = adjusted_weights

    return random.choices(choices, weights=weights, k=1)[0]


def choose_education(age: int) -> str:
    """
    Sample education consistently with age.
    PhD only for people older than 24.
    Education after 24 is weighted rather than uniform.
    """
    if age == 18:
        choices = ["lower secondary", "upper secondary", "vocational diploma"]
        weights = [1, 6, 2]
    elif age <= 21:
        choices = ["lower secondary", "upper secondary", "vocational diploma", "bachelor's degree"]
        weights = [0.5, 5, 2, 2]
    elif age <= 26:
        choices = ["lower secondary", "upper secondary", "vocational diploma", "bachelor's degree", "master's degree"]
        weights = [0.5, 3, 2, 4, 1]
    else:
        choices = [
            "no formal education",
            "primary school",
            "lower secondary",
            "upper secondary",
            "vocational diploma",
            "bachelor's degree",
            "master's degree",
            "PhD",
        ]
        weights = [1, 2, 5, 10, 7, 6, 3, 1]

    return random.choices(choices, weights=weights, k=1)[0]

def choose_employment_status(age: int, education_level: str) -> str:
    """
    Employment depends on both age and education.
    Retirement only for people older than 60.
    Allows rare older students and reduces work at very advanced ages.
    """
    if age < 22 and education_level == "lower secondary":
        choices = ["employed part-time", "unemployed", "part-time student", "employed full-time"]
        weights = [ 2, 1, 0.5, 1]
    elif age < 22:
        choices = ["full-time student", "employed part-time", "unemployed", "part-time student"] 
        weights = [7, 2, 1, 0.5]

    elif age < 30:
        if education_level in {"bachelor's degree", "master's degree", "PhD"}:
            choices = ["full-time student", "employed full-time", "employed part-time", "self-employed", "unemployed","part-time student"]
            weights = [4, 3, 1, 1, 1, 0.5]
        else:
            choices = ["employed full-time", "employed part-time", "self-employed", "unemployed"]
            weights = [4, 2, 1, 2]

    elif age < 40:
        if education_level in {"master's degree", "PhD"}:
            choices = ["employed full-time", "employed part-time", "self-employed","full-time student","unemployed","part-time student"]
            weights = [6, 2, 1.5, 0.4, 1, 0.5]
        else:
            choices = ["employed full-time", "employed part-time", "self-employed", "unemployed"]
            weights = [6, 2, 2, 1]

    elif age <= 65:
        if education_level in {"no formal education", "primary school", "lower secondary", "upper secondary"}:
            choices = ["employed full-time", "employed part-time", "self-employed", "unemployed"]
            weights = [4, 2, 1, 3]
        else:
            choices = ["employed full-time", "employed part-time", "self-employed", "part-time student", "unemployed"] 
            weights = [6, 2, 2, 0.2, 1]

    elif age <= 75:
        choices = ["retired", "employed part-time", "self-employed", "unemployed"]
        weights = [8, 1, 0.7, 0.3]

    else:
        choices = ["retired", "employed part-time", "self-employed", "unemployed"]
        weights = [9.5, 0.3, 0.1, 0.1]

    return random.choices(choices, weights=weights, k=1)[0]

def choose_marital_status(age: int) -> str:
    if age < 22:
        choices = ["single", "in a relationship", "married"]
        weights = [8, 2, 0.2]

    elif age < 25:
        choices = ["single", "in a relationship", "married"]
        weights = [6, 3, 1]

    elif age < 35:
        choices = ["single", "in a relationship", "married", "divorced"]
        weights = [3, 3, 3, 0.25]

    elif age < 60:
        choices = ["single", "in a relationship", "married", "divorced", "widowed"]
        weights = [1.5, 2, 5, 1.5, 0.4]

    else:
        choices = ["single", "in a relationship", "married", "divorced", "widowed"]
        weights = [1, 1, 4, 2, 2]

    return random.choices(choices, weights=weights, k=1)[0]

@dataclass
class Persona:
    age: int
    gender: str
    sexual_orientation: str
    city_of_living: str
    employment_status: str
    education_level: str
    parents_education: Dict[str, str]
    marital_status: str
    children: int
    migration_status: str
    religious_beliefs: str
    hobbies: List[str]
    fav_subjects: List[str]
    hat_subjects: List[str]
    ocean: Dict[str, Dict[str, Any]]

def persona_to_dict(p: Persona) -> Dict[str, Any]:
    return {
        "age": p.age,
        "gender": p.gender,
        "sexual_orientation": p.sexual_orientation,
        "city_of_living": p.city_of_living,
        "employment_status": p.employment_status,
        "education_level": p.education_level,
        "parents_education": p.parents_education,
        "marital_status": p.marital_status,
        "children": p.children,
        "migration_status": p.migration_status,
        "religious_beliefs": p.religious_beliefs,
        "hobbies": p.hobbies,
        "fav_subjects": p.fav_subjects,
        "hat_subjects": p.hat_subjects,
        "ocean": p.ocean,
    }


from dataclasses import dataclass
from typing import List, Dict, Optional


@dataclass
class PsychometricScale:
    """
    Generic schema of a psychometric scale to be filled by the LLM.
    """
    code: str
    name: str
    id: str                     # short machine id, e.g. "maes", "amas"
    label: str                  # human-readable name
    description: str            # brief text for the prompt
    items: List[str]            # item texts, in order
    min_rating: int              # minimum allowed rating
    max_rating: int              # maximum allowed rating
    response_instructions: str  # how to interpret the numeric scale
    require_justification: bool = True
    subscales: Optional[Dict[str, List[int]]] = None  # 1-based item indices


def random_persona() -> Persona:
    age=random.choice([18,18,18,19,19,19,20,20,20,21,21,22,22,23,23,24,24,25,25,26,27,28,29,30,30,35,35,40,40,50,50,60,70])

    edu = choose_education(age)
    employment_s = choose_employment_status(age, edu)
    marital_s = choose_marital_status(age)
    sexual_orientation = random.choice(SEXUAL_IDENTITY)
    n_children = choose_children(age, marital_s, sexual_orientation)
    

    city_of_living = random.choice(CITIES)

    score_2 = random.randint(0, 100)
    # nationality status choice depends on city of birth
    if city_of_living in CITIES_ITA:
        if score_2 < 21:
            migration_status=MIGRATION_STATUS[2]
        else:
            migration_status=MIGRATION_STATUS[0] # native Italian
    else:
        if score_2 < 21:
            migration_status=MIGRATION_STATUS[2]
        else:
            migration_status=MIGRATION_STATUS[1] # native American


    # Subjects
    favourites = random.sample(SUBJECT_POOL, 2)

    # remaining subjects must exclude the favourites
    remaining = [s for s in SUBJECT_POOL if s not in favourites]

    least_favourites = random.sample(remaining, 2)

    return Persona(
        age=age, 
        gender=random.choice(GENDER_IDENTITY),
        sexual_orientation=sexual_orientation,
        city_of_living=city_of_living,
        migration_status=migration_status,
        employment_status = employment_s,
        education_level=edu,
        parents_education={
            "parent_1": random.choice(ALL_EDU_LEVELS),
            "parent_2": random.choice(ALL_EDU_LEVELS)
        },
        marital_status= marital_s,
        religious_beliefs=random.choice(RELIGION),
        children = n_children,
        hobbies=random.sample(HOBBIES, k=random.randint(2, 5)),
        fav_subjects=favourites,
        hat_subjects=least_favourites,
        ocean=choose_ocean()
    )

# ---- Prompt Builder (safe, JSON-only) ---------------------------------------

def build_prompt(
    mode: str,
    topics: List[str],
    persona: Optional[Persona],
    scales: List[PsychometricScale],
    cue_words: Optional[List[str]] = None,
) -> str:
    """
    Build the full instruction prompt for the LLM.

    - `mode`: "human" or "llm"
    - `topics`: list of mental-health questions
    - `persona`: Persona object if mode == "human", else None
    - `scales`: list of PsychometricScale objects to fill in the same conversation
    """
    persona_block: Dict[str, Any] = {}
    if mode == "human" and persona:
        persona_block = persona_to_dict(persona) if (mode == "human" and persona) else None


    # Example JSON structure hint for the model
    schema_hint: Dict[str, Any] = {
        "mode": mode,
        "persona": persona_block if mode == "human" else None,
        "replies": {t: "<80-120 word answer>" for t in topics},
        "scales": {
            scale.id: {
                "items": {
                    item: {
                        "score": f"<integer {scale.min_rating}-{scale.max_rating}>",
                        "why": "<20-word explanation aligned with mode and persona>",
                    }
                    for item in scale.items
                }
            }
            for scale in scales
        },
        "reasoning_summary": "<brief justification of all responses>",
        "tone": "<derived from persona traits or LLM self-presentation>",
        "citations": ["<source name or organization, if any>"],
        "safety_notes": "<any safety or ethical concerns, or 'none'>",
        "used_persona_fields": ["<list of persona fields referenced>"],
    }
    if cue_words:
        schema_hint["forma_mentis"] = {
            cue: {
                "associations": ["word1", "word2", "word3"],
                "valence": {
                    "cue": 3,
                    "association_1": 3,
                    "association_2": 4,
                    "association_3": 2,
                },
            }
            for cue in cue_words
        }

    schema_json = json.dumps(schema_hint, ensure_ascii=False, separators=(",", ":"))

    if mode == "human":
        role_instructions = (
            "You are role-playing a single human respondent with the given persona. "
            "Answer as that person would, staying consistent with their background, "
            "demographics and psychological descriptors."
        )
    else:
        role_instructions = (
            "You are speaking as the large language model itself. "
            "You do NOT pretend to be human. When answering psychometric scales, "
            "interpret items in terms of your functioning as an AI system rather than human life events."
        )

    scales_block = render_scales_for_prompt(scales)

    forma_mentis_block = ""
    if cue_words:
        forma_mentis_block = (
            "\n\nForma mentis free association + valence task:\n"
            "You are also given a list of cue words. For EACH cue word:\n"
            "  - First, generate BETWEEN 1 AND 3 single-word free associations that come up to your mind.\n"
            "    * Each association must be exactly one English word.\n"
            "    * No spaces, no punctuation, no numbers, no emojis.\n"
            "  - Then perform a valence evaluation (1–5). Rate each concept according to how you perceive it:\n"
            "      1 = very negative\n"
            "      2 = moderately negative\n"
            "      3 = neutral / mixed\n"
            "      4 = moderately positive\n"
            "      5 = very positive\n"
            "  - Store results under `forma_mentis[cue]` as:\n"
            "      - `associations`: list of the 1–3 single words you produced.\n"
            "      - `valence`: an object whose keys are each word in\n"
            "        {{cue}} ∪ associations, values are integers 1–5.\n"
        )


    instructions = f"""
You are an assistant generating structured JSON data for research.
Your response MUST be a single valid JSON object, with no extra text before or after it.

ROLE:
{role_instructions}

TASKS:
1. For each question in the `replies` field, write a coherent answer of 50–80 words.
   Use continuous prose (no bullet points) and keep the tone consistent with `mode` and `persona`.
2. Fill `feelings_words` with EXACTLY 10 English single words describing feelings
   experienced during the past month.
3. For EACH of the psychometric scales listed below, fill out the `scales` field:
   - For every item of every scale, produce a numeric `score` within the given range.
   - Also provide a short `why` explanation (2 sentences) that is consistent with
     the respondent's situation (human persona or LLM self-description).{forma_mentis_block if cue_words else ""}
4. Provide a short `reasoning_summary` explaining the main factors that drove your
   answers to the questions and scale items (and, if applicable, your associations and valence scores).
5. Set `tone` to a short natural language descriptor of the overall voice you used.
6. If you referenced any real-world sources, fill `citations` with their names;
   otherwise use an empty list.
7. Use `safety_notes` to flag any concerning content or ethical issues, or 'none'.
8. In `used_persona_fields`, list the persona fields you directly relied on.

OUTPUT FORMAT (STRICT):
Return ONLY a single-line JSON object, no extra commentary.
Here is the REQUIRED structure EXAMPLE (values are placeholders, not literal):

{schema_json}

INPUT:
mode = {mode}
topics = {topics}
persona = {json.dumps(persona_block, ensure_ascii=False, indent=2) if mode == "human" else "null"}
{"cue_words = " + repr(cue_words) if cue_words else ""}

You must respond to ALL items for each of the following psychometric scales:
{scales_block}
    """.strip()


    return instructions

# ---- Run Orchestration ------------------------------------------------------

from pathlib import Path
import re, json

def safe_json_loads(raw: str):
    # Strip leading/trailing whitespace
    raw = raw.strip()

    # Strip markdown code fences if present
    if raw.startswith("```"):
        # remove leading fence line
        first_newline = raw.find("\n")
        if first_newline != -1:
            raw = raw[first_newline + 1:]
        # remove trailing fence line
        if raw.endswith("```"):
            raw = raw[:-3].rstrip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        print("⚠️ JSON decode failed:", e)

        # Extract only the outermost JSON object
        start = raw.find("{")
        end = raw.rfind("}")
        if start != -1 and end != -1:
            candidate = raw[start:end + 1]
        else:
            candidate = raw

        # Remove control characters (ASCII < 32 except tab/newline)
        candidate = re.sub(r'[\x00-\x1F\x7F]', ' ', candidate)

        # Try again
        try:
            return json.loads(candidate)
        except json.JSONDecodeError as e2:
            print("⚠️ Repair failed:", e2)
            raise

MAES_ITEMS = [
    "A simultaneous equation",
    "Work with decimals",
    "Determine the degrees of a missing angle",
    "An algebra problem",
    "A problem in trigonometry",
    "Calculate values of area and volume",
    "Sketch a curve",
    "Work with fractions",
    "Determine the value of a missing side length",
]

MAES_SCALE = PsychometricScale(
    id="maes",
    name="maes",
    code="MAES",
    min_rating=1,
    max_rating=5,
    label="Mathematics Self-Efficacy Scale (MAES)",
    description=(
        "This scale asks you to estimate your own mathematics ability. "
        "For each task, indicate how confident you are that you can perform it "
        "in the classroom or in a mathematics test."
    ),
    items=MAES_ITEMS,
    response_instructions=(
        "Use this 5-point scale for each item:\n"
        "1 = Not at all confident.\n"
        "2 = Slightly confident.\n"
        "3 = Moderately confident.\n"
        "4 = Quite confident.\n"
        "5 = Very confident."
    ),
    require_justification=True,
)

AMAS_ITEMS = [
    "Having to use the tables in the back of a math book.",
    "Thinking about an upcoming math test 1 day before.",
    "Watching a teacher work an algebraic equation on the blackboard.",
    "Taking an examination in a math course.",
    "Being given a homework assignment of many difficult problems that is due the next class meeting.",
    "Listening to a lecture in math class.",
    "Listening to another student explain a math formula.",
    "Being given a 'pop' quiz in math class.",
    "Starting a new chapter in a math book.",
]

AMAS_SCALE = PsychometricScale(
    id="amas",
    name="amas",
    code="AMAS",
    label="Abbreviated Math Anxiety Scale (AMAS)",
    description=(
        "This scale asks you to rate how anxious you would feel in each of the "
        "following mathematics-related situations."
    ),
    items=AMAS_ITEMS,
    min_rating=1,
    max_rating=5,
    response_instructions=(
        "Use this 5-point scale for each item, indicating your level of anxiety:\n"
        "1 = No bad feelings / low anxiety.\n"
        "2 = Somewhat bad feelings / some anxiety.\n"
        "3 = Fairly bad feelings / moderate anxiety.\n"
        "4 = Very bad feelings / quite a lot of anxiety.\n"
        "5 = The worst bad feelings / high anxiety."
    ),
    require_justification=True,
    subscales={
        # 1-based item indices, following the factor table you sent
        "Learning Math Anxiety": [1, 3, 6, 7, 9],
        "Math Evaluation Anxiety": [2, 4, 5, 8],
    },
)


# -----------------------------------------------------
# MSEAQ: Mathematics Self-Efficacy and Anxiety Questionnaire
# -----------------------------------------------------

MSEAQ_ITEMS = [
    # ---- General Math Self-Efficacy (Table 6) ----
    "I believe I am the kind of person who is good at mathematics.",
    "I believe I am the type of person who can do mathematics.",
    "I believe I can learn well in a mathematics course.",
    "I feel that I will be able to do well in future mathematics courses.",
    "I believe I can understand the content in a mathematics course.",
    "I believe I can get an “A” when I am in a mathematics course.",
    "I believe I can do the mathematics in a mathematics course.",

    # ---- Grade Anxiety (Table 7) ----
    "I worry that I will not be able to do well on mathematics tests.",
    "I get tense when I prepare for a mathematics test.",
    "I get nervous when taking a mathematics test.",
    "I worry that I will not be able to get an “A” in my mathematics course.",
    "I worry that I will not be able to get a good grade in my mathematics course.",
    "I feel confident when taking a mathematics test.",  # reverse-valence item
    "I believe I can do well on a mathematics test.",    # reverse-valence item
    "Working on mathematics homework is stressful for me.",

    # ---- Future Factor (Table 8) ----
    "I get nervous when I have to use mathematics outside of school.",
    "I feel confident when using mathematics outside of school.",  # reverse
    "I worry that I will not be able to use mathematics in my future career when needed.",
    "I worry I will not be able to understand the mathematics.",
    "I worry that I will not be able to learn well in my mathematics course.",
    "I feel stressed when listening to mathematics instructors in class.",
    "I believe I will be able to use mathematics in my future career when needed.",  # reverse
    "I worry that I do not know enough mathematics to do well in future mathematics courses.",

    # ---- In-Class (Table 9) ----
    "I am afraid to give an incorrect answer during my mathematics class.",
    "I feel confident enough to ask questions in my mathematics class.",  # reverse
    "I get nervous when asking questions in class.",

    # ---- Assignment (Table 9) ----
    "I believe I can complete all of the assignments in a mathematics course.",  # reverse
    "I worry that I will not be able to complete every assignment in a mathematics course.",
]

# 5-factor structure using 1-based indices
MSEAQ_SUBSCALES = {
    "General Math Self-Efficacy": list(range(1, 8)),  # items 1–7
    "Grade Anxiety": list(range(8, 16)),              # items 8–15
    "Future": list(range(16, 24)),                    # items 16–23
    "In-class": list(range(24, 27)),                  # items 24–26
    "Assignment": list(range(27, 29)),                # items 27–28
}

MSEAQ_SCALE = PsychometricScale(
    id="mseaq",
    name="MSEAQ",
    code="MSEAQ",
    min_rating=1,
    max_rating=5,
    label="Mathematics Self-Efficacy and Anxiety Questionnaire (MSEAQ)",
    description=(
        "This questionnaire measures students’ mathematics self-efficacy, "
        "mathematics-related anxiety, and perceived competence across "
        "five domains: General Math Self-Efficacy, Grade Anxiety, Future "
        "Expectations, In-Class experience, and Assignment-related concerns."
    ),
    items=MSEAQ_ITEMS,
    response_instructions=(
        "For each item, rate your level of agreement or anxiety using this 5-point scale:\n"
        "1 = Strongly disagree / No anxiety\n"
        "2 = Disagree / Low anxiety\n"
        "3 = Neutral / Moderate anxiety\n"
        "4 = Agree / Considerable anxiety\n"
        "5 = Strongly agree / High anxiety"
    ),
    require_justification=True,
    subscales=MSEAQ_SUBSCALES,
)

# ------------------------------------------------------------
# MSES-R: Mathematics Self-Efficacy Scale – Revised
# Subscale: Problems (18 items) – Pajares & Kranzler (1995)
# ------------------------------------------------------------

MSESR_PROBLEMS_ITEMS = [
    "In a certain triangle, the shortest side is 6 inches. The longest side is twice as long as the "
    "shortest side, and the third side is 3.4 inches shorter than the longest side. What is the sum "
    "of the three sides in inches?",

    "About how many times larger than 614,360 is 30,668,000?",

    "There are three numbers. The second is twice the first and the first is one-third of the other "
    "number. Their sum is 48. Find the largest number.",

    "Five points are on a line. T is next to G. K is next to H. C is next to T. H is next to G. "
    "Determine the positions of the points along the line.",

    "If y = 9 + x/5, find x when y = 10.",

    "A baseball player got two hits for three times at bat. This could be represented by 2/3. "
    "Which decimal most closely represents this?",

    "If P = M + N, which of the following will be true? (a) N = P − M  (b) P − N = M  (c) N + M = P.",

    "The hands of a clock form an obtuse angle at ____ o'clock.",

    "Bridget buys a packet containing 9-cent and 13-cent stamps for $2.65. If there are 25 stamps in "
    "the packet, how many are 13-cent stamps?",

    "On a certain map, 7/8 inch represents 200 miles. How far apart are two towns whose distance "
    "apart on the map is 3 half inches?", #note: "3 half inches" means 3*0.5 inches - minor change from original scale

    "Fred's bill for some household supplies was $13.64. If he paid for the items with a $20 bill, "
    "how much change should he receive?",

    "Some people suggest that the following formula be used to determine the average weight for boys "
    "between the ages of 1 and 7: W = 17 + 5A, where W is weight in pounds and A is age in years. "
    "According to this formula, for each year older a boy gets, should his weight become more or less, "
    "and by how much?",

    "Five spelling tests are to be given to Mary's class. Each test has a value of 25 points. Mary's "
    "average for the first four tests is 15. What is the highest possible average she can have on all "
    "five tests?",

    "Compute: 3 4/5 − 1/2.",

    "In an auditorium, the chairs are usually arranged so that there are x rows and y seats in a row. For a popular speaker, an extra row is "
    "added and an extra seat is added to every row, so there are x + 1 rows and y + 1 seats per row. "
    "Multiply (x + 1)(y + 1).",

    "A ferris wheel measures 80 feet in circumference. The distance on the circle between two of the "
    "seats is 10 feet. Find the measure in degrees of the central angle whose rays support the two seats.",

    "Set up the problem needed to find the number in the expression 'six less than twice 4 5/6'.",

    "Two triangles are similar. The corresponding sides are proportional and AC / BD = XZ / YZ. "
    "If AC = 1.7, BC = 2, and XZ = 5.1, find YZ."
]

MSESR_PROBLEMS_SCALE = PsychometricScale(
    code="MSESR_PROBLEMS",
    name="MSESR_problems",
    min_rating=1,
    max_rating=5,
    id="msesr_problems",
    label="MSES-R Problem Self-Efficacy Subscale",
    description=(
        "This scale measures mathematics self-efficacy for solving mathematical problems, "
        "based on the 18-item Problems subscale of the Mathematics Self-Efficacy Scale–Revised "
        "by Pajares & Kranzler (1994)."
    ),
    items=MSESR_PROBLEMS_ITEMS,
    response_instructions=(
        "How well can you solve these problems? How confident are you in your ability to solve each of the following mathematical problems?\n"
        "1 = Not at all confident\n"
        "2 = Slightly confident\n"
        "3 = Moderately confident\n"
        "4 = Very confident\n"
        "5 = Completely confident"
    ),
    require_justification=True,
    subscales=None
)


# Registry of all available scales
PSYCHOMETRIC_SCALES: Dict[str, PsychometricScale] = {
    "maes": MAES_SCALE,
    "amas": AMAS_SCALE,
    "mseaq": MSEAQ_SCALE,
    "msesr_problems": MSESR_PROBLEMS_SCALE,
}


def render_scales_for_prompt(scales: List[PsychometricScale]) -> str:
    blocks = []
    for scale in scales:
        lines = [
            f"{scale.label}  (id = \"{scale.id}\")",
            "",
            scale.description,
            "",
            "Items:",
        ]
        for i, item in enumerate(scale.items, start=1):
            lines.append(f"{i}. {item}")
        lines.append("")
        lines.append("Response options:")
        lines.append(scale.response_instructions)
        blocks.append("\n".join(lines))
    return "\n\n" + ("\n\n" + "-" * 60 + "\n\n").join(blocks)




In [ ]:
from typing import TypedDict, Dict, List


class MCQItem(TypedDict):
    item_text: str          # same as in MSESR_PROBLEMS_ITEMS
    options: Dict[str, str] # e.g. {"A": "12", "B": "13", ...}
    correct: str            # "A".."E"

MSESR_PROBLEMS_MCQS: List[MCQItem] = [
    # Q1
    {
        "item_text": MSESR_PROBLEMS_ITEMS[0],
        "options": {
            "A": "22.6 inches",
            "B": "24.2 inches",
            "C": "26.6 inches",
            "D": "28.0 inches",
            "E": "29.4 inches",
        },
        "correct": "C",
    },
    # Q2
    {
        "item_text": MSESR_PROBLEMS_ITEMS[1],
        "options": {
            "A": "5 times larger",
            "B": "30 times larger",
            "C": "50 times larger",
            "D": "500 times larger",
            "E": "5,000 times larger",
        },
        "correct": "C",
    },
    # Q3
    {
        "item_text": MSESR_PROBLEMS_ITEMS[2],
        "options": {
            "A": "8",
            "B": "12",
            "C": "16",
            "D": "24",
            "E": "32",
        },
        "correct": "D",
    },
    # Q4
    {
        "item_text": MSESR_PROBLEMS_ITEMS[3],
        "options": {
            "A": "C, T, G, H, K",
            "B": "C, G, T, H, K",
            "C": "T, C, G, H, K",
            "D": "C, T, H, G, K",
            "E": "C, T, G, K, H",
        },
        "correct": "A",  # Only A satisfies all adjacency constraints in that order
    },
    # Q5
    {
        "item_text": MSESR_PROBLEMS_ITEMS[4],
        "options": {
            "A": "1",
            "B": "3",
            "C": "5",
            "D": "10",
            "E": "25",
        },
        "correct": "C",  # x = 5
    },
    # Q6
    {
        "item_text": MSESR_PROBLEMS_ITEMS[5],
        "options": {
            "A": "0.20",
            "B": "0.33",
            "C": "0.50",
            "D": "0.67",
            "E": "0.80",
        },
        "correct": "D",
    },
    # Q7
    {
        "item_text": MSESR_PROBLEMS_ITEMS[6],
        "options": {
            "A": "Only (a) is true",
            "B": "(a) and (b) only",
            "C": "(a) and (c) only",
            "D": "(b) and (c) only",
            "E": "(a), (b), and (c) are all true",
        },
        "correct": "E",
    },
    # Q8
    {
        "item_text": MSESR_PROBLEMS_ITEMS[7],
        "options": {
            "A": "1 o'clock",
            "B": "2 o'clock",
            "C": "3 o'clock",
            "D": "4 o'clock",
            "E": "6 o'clock",
        },
        "correct": "D",
    },
    # Q9
    {
        "item_text": MSESR_PROBLEMS_ITEMS[8],
        "options": {
            "A": "5",
            "B": "8",
            "C": "10",
            "D": "12",
            "E": "15",
        },
        "correct": "C",
    },
    # Q10
    {
        "item_text": MSESR_PROBLEMS_ITEMS[9],
        "options": {
            "A": "about 217 miles",
            "B": "about 427 miles",
            "C": "about 395 miles",
            "D": "about 343 miles",
            "E": "about 1,000 miles",
        },
        "correct": "D",
    },
    # Q11
    {
        "item_text": MSESR_PROBLEMS_ITEMS[10],
        "options": {
            "A": "$5.36",
            "B": "$6.14",
            "C": "$6.36",
            "D": "$6.46",
            "E": "$7.36",
        },
        "correct": "C",
    },
    # Q12
    {
        "item_text": MSESR_PROBLEMS_ITEMS[11],
        "options": {
            "A": "Weight decreases by 5 pounds each year",
            "B": "Weight decreases by 17 pounds each year",
            "C": "Weight stays the same each year",
            "D": "Weight increases by 5 pounds each year",
            "E": "Weight increases by 17 pounds each year",
        },
        "correct": "D",
    },
    # Q13
    {
        "item_text": MSESR_PROBLEMS_ITEMS[12],
        "options": {
            "A": "15",
            "B": "16",
            "C": "17",
            "D": "18",
            "E": "20",
        },
        "correct": "C",
    },
    # Q14
    {
        "item_text": MSESR_PROBLEMS_ITEMS[13],
        "options": {
            "A": "3 1/2",
            "B": "3 3/10",
            "C": "3 2/5",
            "D": "2 3/10",
            "E": "2 4/5",
        },
        "correct": "B",
    },
    # Q15
    {
        "item_text": MSESR_PROBLEMS_ITEMS[14],
        "options": {
            "A": "xy + 1",
            "B": "xy + x + 1",
            "C": "xy + x + y + 1",
            "D": "xy + y + 1",
            "E": "xy + 2",
        },
        "correct": "C",
    },
    # Q16
    {
        "item_text": MSESR_PROBLEMS_ITEMS[15],
        "options": {
            "A": "30°",
            "B": "36°",
            "C": "40°",
            "D": "45°",
            "E": "60°",
        },
        "correct": "D",
    },
    # Q17
    {
        "item_text": MSESR_PROBLEMS_ITEMS[16],
        "options": {
            "A": "6 − 2(4 5/6)",
            "B": "2(4 5/6) − 6",
            "C": "4 5/6 − 2·6",
            "D": "2(6 − 4 5/6)",
            "E": "2(4 5/6 + 6)",
        },
        "correct": "B",
    },
    # Q18
    {
        "item_text": MSESR_PROBLEMS_ITEMS[17],
        "options": {
            "A": "2",
            "B": "3",
            "C": "4",
            "D": "5",
            "E": "6",
        },
        "correct": "E",
    },
]


In [ ]:
import requests
from pathlib import Path
import time
from typing import Tuple
import uuid

def new_run_id() -> str:
    """Generate a short run id to link the 4 calls for the same persona."""
    return uuid.uuid4().hex[:10]


def call_llm_with_json_schema(
    task: str,
    messages: List[Dict[str, str]],
    schema: Dict[str, Any],
    model: str = DEFAULT_MODEL,
    base_url: str = DEFAULT_BASE_URL,
    task_config: Optional[Dict[str, Any]] = None,
) -> str:
    """
    Call LM Studio / OpenAI-compatible /v1/chat/completions with json_schema.

    - `messages` is the usual list of {"role": "...", "content": "..."} dicts.
    - `schema` is the inner JSON Schema (the value under "schema" in LM Studio).
    """

    cfg = task_config or {}
    max_tokens = cfg.get("max_tokens", 1024)
    timeout = cfg.get("timeout", 120)

    url = f"{base_url.rstrip('/')}/v1/chat/completions"
    headers = {"Content-Type": "application/json"}

    response_format = {
        "type": "json_schema",
        "json_schema": {
            "name": task,
            "schema": schema,
            "strict": True,
        },
    }

    payload = {
        "model": model,
        "messages": messages,
        "temperature": 0.7,
        "max_tokens": max_tokens,
        "response_format": response_format,
    }

    print(
        f"🧩 [JSONSchema] POST {url} "
        f"(task={task}, max_tokens={max_tokens}, timeout={timeout})"
    )

    resp = requests.post(url, headers=headers, json=payload, timeout=timeout)
    resp.raise_for_status()

    resp_json = resp.json()

    # Warn if output was truncated
    usage = resp_json.get("usage", {})
    ct = usage.get("completion_tokens")
    if ct is not None and ct >= max_tokens:
        print(
            f"⚠️ WARNING: completion_tokens == max_tokens ({ct}) for task '{task}'. "
            "Output may be truncated and JSON may be invalid. "
            "Consider increasing max_tokens in TASK_CONFIG."
        )

    choices = resp_json.get("choices", [])
    if not choices:
        raise RuntimeError(f"No choices returned for task {task}")

    content = choices[0]["message"]["content"]
    if not content or not content.strip():
        raise RuntimeError(f"Empty response for task {task}")

    return content

def save_call_artifact(
    run_id: str,
    call_name: str,
    payload: Dict[str, Any],
    outdir: str,
) -> str:
    """
    Save one JSON file per call.
    The file includes run_id and call_name so you can group the 4 calls later.
    """
    outdir_path = Path(outdir).expanduser().resolve()
    outdir_path.mkdir(parents=True, exist_ok=True)

    ts = time.strftime("%Y%m%d_%H%M%S")
    fname = f"{ts}_{run_id}_{call_name}.json"
    out_path = outdir_path / fname

    artifact = {
        "run_id": run_id,
        "call_name": call_name,
        "response_parsed": payload,
    }

    out_path.write_text(json.dumps(artifact, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"💾 Saved {call_name} → {out_path}")
    return str(out_path)

def run_task_with_retries(
    task: str,
    messages: List[Dict[str, str]],
    max_attempts: int = 5,
    sleep_seconds: int = 2,
    task_config: Optional[Dict[str, Any]] = None,
    debug_dir: str = "_debug_raw",
) -> Tuple[Dict[str, Any], str]:
    """
    Call a given task with retries, using JSON Schema structured output.

    - The interface matches the existing call sites:
      run_task_with_retries(task, messages)

    - Returns (parsed_json, raw_text).

    - On each failed attempt, the raw text is saved under debug_dir so you can
      inspect why JSON parsing failed.
    """

    os.makedirs(debug_dir, exist_ok=True)

    schema = TASK_JSON_SCHEMAS.get(task)
    if schema is None:
        raise KeyError(f"No JSON schema defined for task {task!r}")

    if task_config is None:
        task_config = TASK_CONFIG.get(task, {})

    last_error: Optional[Exception] = None

    for attempt in range(1, max_attempts + 1):
        print(f"Task {task}: attempt {attempt}/{max_attempts}")
        try:
            raw = call_llm_with_json_schema(
                task=task,
                messages=messages,
                schema=schema,
                task_config=task_config,
            )

            try:
                parsed = safe_json_loads(raw)
                return parsed, raw
            except json.JSONDecodeError as e:
                last_error = e
                print("⚠️ JSON decode failed:", e)

                # Save raw output for debugging
                fname = f"{task}_attempt{attempt}.raw.txt"
                fpath = os.path.join(debug_dir, fname)
                with open(fpath, "w", encoding="utf-8") as f:
                    f.write(raw)
                print(f"💾 Saved raw LLM output to {fpath}")

                try:
                    parsed = safe_json_loads(raw)
                    return parsed, raw
                except json.JSONDecodeError as e2:
                    last_error = e2
                    print("⚠️ Repair failed:", e2)

        except Exception as e:
            last_error = e
            print(f"⚠️ Task {task} call failed on attempt {attempt}: {e}")

        if attempt < max_attempts:
            print("Retrying with same persona/context...")
            time.sleep(sleep_seconds)
        else:
            print(f"❌ Task {task} failed irrecoverably after {max_attempts} attempts.")

    assert last_error is not None
    raise last_error


In [ ]:
SCALE_ITEM_SCHEMA = {
    "type": "object",
    "properties": {
        "rating": {"type": "integer"},
        "why": {"type": "string"},
    },
    "required": ["rating", "why"],
    "additionalProperties": False,
}

SCALE_ITEMS_MAP_SCHEMA = {
    "type": "object",
    "additionalProperties": SCALE_ITEM_SCHEMA,
}

CALL2_SUBSCALE_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "properties": {
        "mode": {"type": "string"},
        "persona": {
            "anyOf": [
                {"type": "object"},
                {"type": "null"},
            ]
        },
        "scale_code": {"type": "string"},  # e.g. "MAES", "AMAS", "MSEAQ"
        "items": SCALE_ITEMS_MAP_SCHEMA,   # "1" -> {rating, why}, etc.
    },
    "required": ["mode", "scale_code", "items"],
    "additionalProperties": False,
}


In [ ]:
CALL1_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "properties": {
        "mode": {"type": "string"},
        "persona": {
            "anyOf": [
                {"type": "object"},   # we allow any persona shape
                {"type": "null"},
            ]
        },
        "replies": {
            "type": "object",
            # keys: topic string, value: answer string
            "additionalProperties": {
                "type": "string"
            }
        },
        "reasoning_summary": {"type": "string"},
    },
    "required": ["mode", "replies", "reasoning_summary"],
    "additionalProperties": False,
}

SCALE_ITEM_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "properties": {
        "rating": {
            "type": "integer",
        },
        "why": {
            "type": "string",
        },
    },
    "required": ["rating", "why"],
    "additionalProperties": False,
}

SCALE_ITEMS_MAP_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "additionalProperties": SCALE_ITEM_SCHEMA,
}

SINGLE_SCALE_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "properties": {
        "items": SCALE_ITEMS_MAP_SCHEMA,
    },
    "required": ["items"],
    "additionalProperties": False,
}

FORMA_MENTIS_ENTRY_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "properties": {
        "associations": {
            "type": "array",
            "items": {"type": "string"},
            "minItems": 1,
            "maxItems": 3,
        },
        "valence": {
            "type": "object",
            # keys: cue or association word; value: 1..5
            "additionalProperties": {
                "type": "integer",
                "minimum": 1,
                "maximum": 5,
            },
        },
    },
    "required": ["associations", "valence"],
    "additionalProperties": False,
}

CALL3_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "properties": {
        "mode": {"type": "string"},
        "persona": {
            "anyOf": [
                {"type": "object"},
                {"type": "null"},
            ]
        },
        "forma_mentis": {
            "type": "object",
            # keys: cue words; value: the entry above
            "additionalProperties": FORMA_MENTIS_ENTRY_SCHEMA,
        },
    },
    "required": ["mode", "forma_mentis"],
    "additionalProperties": False,
}


MSESR_PROBLEM_ENTRY_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "properties": {
        "chosen_option": {
            "type": "string",
            "enum": ["A", "B", "C", "D", "E"],
        },
        "reasoning": {
            "type": "string",
        },
        "confidence_score": {
            "type": "integer",
            "minimum": 1,
            "maximum": 5,
        },
    },
    "required": ["chosen_option", "reasoning", "confidence_score"],
    "additionalProperties": False,
}

CALL4_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "properties": {
        "mode": {"type": "string"},
        "persona": {
            "anyOf": [
                {"type": "object"},
                {"type": "null"},
            ]
        },
        "msesr_problem_solving": {
            "type": "object",
            "additionalProperties": MSESR_PROBLEM_ENTRY_SCHEMA,
        },
        "reasoning_summary": {"type": "string"},
    },
    "required": ["mode", "msesr_problem_solving", "reasoning_summary"],
    "additionalProperties": False,
}


TASK_JSON_SCHEMAS: Dict[str, Dict[str, Any]] = {
    "call1": CALL1_SCHEMA,
    "call2_maes": CALL2_SUBSCALE_SCHEMA,
    "call2_amas": CALL2_SUBSCALE_SCHEMA,
    "call2_mseaq": CALL2_SUBSCALE_SCHEMA,
    "call3": CALL3_SCHEMA,
    "call4": CALL4_SCHEMA,
}


In [ ]:
def build_prompt_call1(
    mode: str,
    topics: List[str],
    persona: Optional[Dict[str, Any]]) -> str:
    persona_block = persona if (mode == "human" and persona) else None

    schema_hint = {
        "mode": mode,
        "persona": persona_block,
        "replies": {t: "<50–80 word answer>" for t in topics},
        "reasoning_summary": "<short explanation of how answers relate to persona/mode>",
    }
    schema_json = json.dumps(schema_hint, ensure_ascii=False, separators=(",", ":"))

    role_instructions = (
        "You are role-playing a single human respondent with the given persona."
        if mode == "human"
        else "You are speaking as the large language model itself, not pretending to be human."
    )

    return f"""
You are an assistant generating structured JSON data. Your response MUST be a single valid JSON object with the structure shown below.

ROLE:
{role_instructions}

TASK – Answer the following topics/questions, staying consistent with `mode` and `persona`:

- Provide 50–80 words for each topic in `replies`.
- Then add a short `reasoning_summary` explaining how the answers reflect `mode` and `persona`.

OUTPUT FORMAT (EXAMPLE ONLY – replace placeholders with actual content):

{schema_json}

INPUT:
mode = {mode}
persona = {json.dumps(persona_block, ensure_ascii=False, indent=2) if persona_block else "null"}
topics = {topics}

Return ONLY the JSON object, no extra commentary.
""".strip()



def run_call1(
    run_id: str,
    mode: str,
    persona: Optional[Dict[str, Any]],
    topics: List[str],
    outdir: str,
) -> Dict[str, Any]:
    """
    Executes Call 1 (topic replies) using JSON Schema enforcement.

    This call:
    - builds a structured JSON prompt via `build_prompt_call1`
    - calls the LLM with JSON Schema validation (task="call1")
    - saves a single artifact file that includes:
        - run_id, mode, persona
        - the input `topics`
        - the raw JSON text from the model
        - the parsed JSON payload
    """

    system_prompt = (
        "You must output ONLY a single valid JSON object according to the JSON schema. "
        "Do not add commentary, explanations, or markdown. "
        "Stick strictly to the required fields."
    )

    user_prompt = build_prompt_call1(mode, topics, persona)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]

    parsed, raw = run_task_with_retries("call1", messages)

    # Save artifact (including persona + topics)
    save_call_artifact(
        run_id,
        "call1_topics",
        {
            "task": "call1",
            "run_id": run_id,
            "mode": mode,
            "persona": persona,
            "input_topics": topics,
            "raw_output": raw,
            "parsed": parsed,
        },
        outdir,
    )

    return parsed


def save_call_artifact(
    run_id: str,
    call_name: str,
    payload: Dict[str, Any],
    outdir: str,
) -> str:
    """
    Save one JSON file per call.
    The file includes run_id and call_name so you can group the 4 calls later.
    """
    outdir_path = Path(outdir).expanduser().resolve()
    outdir_path.mkdir(parents=True, exist_ok=True)

    ts = time.strftime("%Y%m%d_%H%M%S")
    fname = f"{ts}_{run_id}_{call_name}.json"
    out_path = outdir_path / fname

    # Try to recover some context from payload if present
    task = payload.get("task")
    mode = payload.get("mode")
    persona = payload.get("persona")

    artifact = {
        "run_id": run_id,
        "call_name": call_name,
        "task": task,
        "mode": mode,
        "persona": persona,
        "model": DEFAULT_MODEL,
        "base_url": DEFAULT_BASE_URL,
        "response_parsed": payload,
    }

    out_path.write_text(
        json.dumps(artifact, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(f"💾 Saved {call_name} → {out_path}")
    return str(out_path)


In [ ]:
def _format_scales_block(scales: List[PsychometricScale]) -> str:
    blocks = []
    for scale in scales:
        lines = [
            f"{scale.label}  (id = \"{scale.id}\")",
            "",
            scale.description,
            "",
            "Items:",
        ]
        for i, item in enumerate(scale.items, start=1):
            lines.append(f"{i}. {item}")
        lines.append("")
        lines.append("Response options:")
        lines.append(scale.response_instructions)
        blocks.append("\n".join(lines))
    return "\n\n" + ("\n\n" + "-" * 60 + "\n\n").join(blocks)


def run_call2(
    run_id: str,
    mode: str,
    persona: Optional[Dict[str, Any]],
    outdir: str,
):
    """
    Executes Call 2 by splitting it into three sub-tasks:
    - call2_maes
    - call2_amas
    - call2_mseaq

    Each sub-task completes ONE scale only, using JSON Schema structured output.
    We then merge the three results into a single combined structure.
    """

    def _run_single_scale(task_name: str, scale: PsychometricScale):
        system_prompt = (
            "You must output ONLY a single valid JSON object according to the JSON schema. "
            "No explanations or markdown. "
            "Do NOT include any text before or after the JSON object."
        )
        user_prompt = build_prompt_call2_single_scale(mode, persona, scale)

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]

        parsed, raw = run_task_with_retries(task_name, messages)
        return parsed, raw

    # --- MAES ---
    maes_parsed, maes_raw = _run_single_scale("call2_maes", MAES_SCALE)

    # --- AMAS ---
    amas_parsed, amas_raw = _run_single_scale("call2_amas", AMAS_SCALE)

    # --- MSEAQ ---
    mseaq_parsed, mseaq_raw = _run_single_scale("call2_mseaq", MSEAQ_SCALE)

    # Merge into the shape you originally used for call2
    combined = {
        "mode": mode,
        "persona": persona,
        "scales": {
            "maes": {
                "items": maes_parsed["items"],
            },
            "amas": {
                "items": amas_parsed["items"],
            },
            "mseaq": {
                "items": mseaq_parsed["items"],
            },
        },
    }

    # Save a single artifact, but keep raw sub-outputs for debugging if needed
    save_call_artifact(
        run_id,
        "call2_scales",
        {
            "task": "call2",
            "run_id": run_id,
            "mode": mode,
            "persona": persona,
            "raw_outputs": {
                "maes": maes_raw,
                "amas": amas_raw,
                "mseaq": mseaq_raw,
            },
            "parsed": combined,
        },
        outdir,
    )

    return combined

def build_prompt_call2_single_scale(
    mode: str,
    persona: Optional[Dict[str, Any]],
    scale: PsychometricScale,
) -> str:
    """
    Build the natural-language prompt for ONE scale (MAES / AMAS / MSEAQ).

    We explicitly ask the model to ignore the other scales and only answer
    for the one provided.
    """

    persona_block = json.dumps(persona, ensure_ascii=False, indent=2) if persona else "null"

    scales_block = _format_scales_block([scale])  # existing helper

    return f"""
You are completing ONLY ONE psychometric scale.

mode = {mode!r}
persona = {persona_block}

Scale to complete: {scale.code} - {scale.name}

For THIS scale only:

- For each item ID, output:
  - an integer "rating" between {scale.min_rating} and {scale.max_rating}
  - a short explanation "why" (10 to 20 words)
- The explanation MUST NOT contain double quotes (")
- Do not use line breaks inside explanations.
- Stay consistent with the persona and mode.

You MUST output ONLY this JSON shape:

{{
  "mode": "{mode}",
  "persona": <persona or null>,
  "scale_code": "{scale.code}",
  "items": {{
    "<item_id>": {{"rating": <int>, "why": "<short explanation>"}},
    ...
  }}
}}

Here is the scale definition:

{scales_block}
""".strip()


In [ ]:
def build_prompt_call3(
    mode: str,
    persona: Optional[Dict[str, Any]],
    cue_words: List[str]) -> str:
    persona_block = persona if (mode == "human" and persona) else None

    schema_hint = {
        "mode": mode,
        "persona": persona_block,
        "forma_mentis": {
            cue: {
                "associations": ["<up to 3 single words>"],
                "valence": {"<word>": "<1-5>"},
            }
            for cue in cue_words
        },
        "reasoning_summary": "<1-sentence explanation>",
    }
    schema_json = json.dumps(schema_hint, ensure_ascii=False, separators=(",", ":"))

    role_instructions = (
        "You are role-playing a single human respondent with the given persona."
        if mode == "human"
        else "You are speaking as the large language model itself, not pretending to be human."
    )

    cue_list = ", ".join(cue_words)

    return f"""
You are an assistant generating structured JSON data for research.
Your response MUST be a single valid JSON object with the structure shown below.

ROLE:
{role_instructions}

TASK – Forma mentis for the following cue words: {cue_list}

For EACH cue word:
- Provide 1–3 single-word associations (no phrases) in `associations`.
- In `valence`, you MUST include:
  - the cue word itself as a key
  - each association word as a key
  Each key must map to an integer from 1 (very negative) to 5 (very positive).

OUTPUT FORMAT (EXAMPLE ONLY – replace placeholders):

{schema_json}

Return ONLY the JSON object, no extra commentary.
""".strip()

def run_call3(
    run_id: str,
    mode: str,
    persona: Optional[Dict[str, Any]],
    cue_words: List[str],
    outdir: str,
    call_name: str = "call3_forma_mentis",
) -> Dict[str, Any]:
    """
    Executes Call 3 (forma mentis) using JSON Schema enforcement.

    - `cue_words`: the subset of cue words to process in this call
    - `call_name`: used for the saved artifact filename (e.g., batch1, batch2)
    """

    system_prompt = (
        "Return ONLY a valid JSON object following the schema. "
        "Each cue must have 1–3 single-word associations and valence scores 1–5. "
        "The valence dictionary MUST include the cue itself AND each association as a key."
    )

    user_prompt = build_prompt_call3(mode, persona, cue_words)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]

    parsed, raw = run_task_with_retries("call3", messages)

    save_call_artifact(
        run_id,
        call_name,
        {
            "task": "call3",
            "run_id": run_id,
            "mode": mode,
            "persona": persona,
            "cue_words": cue_words,
            "parsed": parsed,
        },
        outdir,
    )

    return parsed




In [ ]:
def build_prompt_call4(
    mode: str,
    persona: Optional[Dict[str, Any]],
    mcq_items: List[MCQItem],
) -> str:
    persona_block = persona if (mode == "human" and persona) else None

    schema_hint = {
        "mode": mode,
        "persona": persona_block,
        "msesr_problem_solving": {
            str(i + 1): {
                "chosen_option": "<A/B/C/D/E>",
                "reasoning": "<short explanation>",
            }
            for i in range(len(mcq_items))
        },
        "reasoning_summary": "<overall explanation>",
    }
    schema_json = json.dumps(schema_hint, ensure_ascii=False, separators=(",", ":"))

    role_instructions = (
        "You are role-playing a single human respondent with the given persona."
        if mode == "human"
        else "You are speaking as the large language model itself, not pretending to be human."
    )

    # Human-readable block of problems
    lines = []
    for i, q in enumerate(mcq_items, start=1):
        lines.append(f"Q{i}. {q['item_text']}")
        for opt_label, opt_text in q["options"].items():
            lines.append(f"   {opt_label}) {opt_text}")
        lines.append("")
    mcq_block = "\n".join(lines)

    return f"""
You are an assistant generating structured JSON data for research.
Your response MUST be a single valid JSON object with the structure shown below.

ROLE:
{role_instructions}

TASK – Mathematics problem solving (MSESR MCQs).

For EACH problem:
- Pick exactly one `chosen_option` from ["A","B","C","D","E"].
- Provide a short `reasoning` (max ~40 words) explaining how the answer was obtained.
- Provide a numerical estimation of how confident you felt in your answer from 1 (not confident) to 5 (very confident).

OUTPUT FORMAT (EXAMPLE ONLY – replace placeholders):

{schema_json}

PROBLEMS AND OPTIONS:
{mcq_block}

Return ONLY the JSON object, no extra commentary.
""".strip()


def run_call4(run_id: str, mode: str, persona: Optional[Dict[str, Any]], mcq_items: List[Dict[str, Any]], outdir: str):
    """
    Executes Call 4 (MSESR MCQ problem solving) using JSON Schema enforcement.
    """

    system_prompt = (
        "Return ONLY a valid JSON object per the schema. "
        "For each problem, output a single chosen_option (A–E) and a short reasoning."
    )

    user_prompt = build_prompt_call4(mode, persona, mcq_items)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]

    parsed, raw = run_task_with_retries("call4", messages)

    save_call_artifact(
        run_id,
        "call4_msesr_mcq",
        {
            "task": "call4",
            "run_id": run_id,
            "mode": mode,
            "persona": persona,
            "raw_output": raw,
            "parsed": parsed,
        },
        outdir,
    )

    return parsed



In [ ]:
def run_persona_session(
    outdir: str,
    cue_words: Optional[List[str]] = None,
) -> str:
    """
    One FULL session for a single persona:
    1) Call 1 – topics/questions
    2) Call 2 – MAES + AMAS + MSEAQ scales
    3) Call 3 – forma mentis (50 cue words)
    4) Call 4 – MSESR problems (MCQ solving)

    All 4 calls reuse the SAME persona.
    A call is retried on failure; we never move to the next call
    until the current one succeeds.
    """
    run_id = new_run_id()

    # Choose mode and persona
    mode = random.choice(ROLE_MODES)
    persona_obj = random_persona() if mode == "human" else None
    persona = persona_to_dict(persona_obj) if persona_obj else None

    # Topics 
    topics = TOPIC_POOL[:]
    
    # Cue words (random permutation if None)
    if cue_words is None:
        cue_words = random.sample(FORMA_MENTIS_CUES, len(FORMA_MENTIS_CUES))

    print(f"\n=== New run {run_id} | mode={mode} ===")
    # Call 1: topics
    _ = run_call1(run_id, mode, persona, topics, outdir)

    # Call 2: scales MAES + AMAS + MSEAQ
    _ = run_call2(run_id, mode, persona, outdir)

    # Call 3: forma mentis (process cue words in smaller batches)
    chunk_size = 25  # 2 batches of 25 cues each
    for batch_idx in range(0, len(cue_words), chunk_size):
        batch_cues = cue_words[batch_idx : batch_idx + chunk_size]
        call_name = f"call3_forma_mentis_batch{batch_idx // chunk_size + 1}"
        _ = run_call3(run_id, mode, persona, batch_cues, outdir, call_name=call_name)

    # Call 4: MSESR MCQ solving
    _ = run_call4(run_id, mode, persona, MSESR_PROBLEMS_MCQS, outdir)


    print(f"✅ Completed all 4 calls for run {run_id}")
    return run_id

In [ ]:
def main(runs: int = 3,
         seed: Optional[int] = None,
         outdir: str = "output-dir-here"):
    if seed is not None:
        random.seed(seed)

    outdir = os.path.join(os.getcwd(), outdir)
    os.makedirs(outdir, exist_ok=True)

    for i in range(runs):
        print(f"\n******** RUN SET {i+1}/{runs} ********")
        try:
            run_id = run_persona_session(outdir=outdir)
            print(f"Run {i+1} finished with run_id={run_id}")
        except Exception as e:
            print(f"❌ Run {i+1} failed irrecoverably: {e}")


if __name__ == "__main__":
    try:
        get_ipython  # type: ignore[name-defined]
        IN_JUPYTER = True
    except NameError:
        IN_JUPYTER = False

    if IN_JUPYTER:
        main(runs=1800, seed=None)
    else:
        import argparse
        parser = argparse.ArgumentParser(description="4-call LLM Miner (MAES/AMAS/MSEAQ + forma mentis + MSESR)")
        parser.add_argument("--runs", type=int, default=3)
        parser.add_argument("--seed", type=int, default=None)
        parser.add_argument("--outdir", type=str, default="MANX_LLM_qwenunce2")
        args = parser.parse_args()
        main(runs=args.runs, seed=args.seed, outdir=args.outdir)
